In [7]:
import json
from pathlib import Path

import pandas as pd

FILE_PATH = "data/News_Category_Dataset_v3.json"

TOP_CATEGORIES = [
    "POLITICS",
    "WELLNESS",
    "ENTERTAINMENT",
    "TRAVEL",
    "STYLE & BEAUTY",
    "PARENTING",
    "HEALTHY LIVING",
    "QUEER VOICES",
    "FOOD & DRINK",
    "BUSINESS",
    "COMEDY",
    "SPORTS",
    "BLACK VOICES",
    "HOME & LIVING",
    "PARENTS",
]

In [8]:
def read_rows(path):
    path = Path(path)
    rows = []
    with path.open(encoding="utf-8") as source:
        for line_number, line in enumerate(source, 1):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue
            if not isinstance(row, dict):
                continue
            rows.append((line_number, row))
    return rows

rows = read_rows(FILE_PATH)
df = pd.DataFrame([row for _, row in rows])
df.head()

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 209527 entries, 0 to 209526
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   link               209527 non-null  str  
 1   headline           209527 non-null  str  
 2   category           209527 non-null  str  
 3   short_description  209527 non-null  str  
 4   authors            209527 non-null  str  
 5   date               209527 non-null  str  
dtypes: str(6)
memory usage: 9.6 MB


In [10]:
normalized_categories = (
    df["category"].str.normalize("NFKC").str.replace(r"\s+", " ", regex=True).str.strip().str.upper()
)
scoped_df = df.loc[normalized_categories.isin(TOP_CATEGORIES)].copy()
scoped_df.info()

<class 'pandas.DataFrame'>
Index: 148122 entries, 2 to 209526
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   link               148122 non-null  str  
 1   headline           148122 non-null  str  
 2   category           148122 non-null  str  
 3   short_description  148122 non-null  str  
 4   authors            148122 non-null  str  
 5   date               148122 non-null  str  
dtypes: str(6)
memory usage: 7.9 MB


In [11]:
scoped_df = scoped_df.replace(r"(?i)^\s*(?:null)?\s*$", pd.NA, regex=True)
scoped_df.info()

<class 'pandas.DataFrame'>
Index: 148122 entries, 2 to 209526
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   link               148122 non-null  str  
 1   headline           148119 non-null  str  
 2   category           148122 non-null  str  
 3   short_description  136608 non-null  str  
 4   authors            123706 non-null  str  
 5   date               148122 non-null  str  
dtypes: str(6)
memory usage: 7.9 MB


In [12]:
scoped_df = scoped_df.dropna().copy()
scoped_df.info()

<class 'pandas.DataFrame'>
Index: 114452 entries, 2 to 209512
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   link               114452 non-null  str  
 1   headline           114452 non-null  str  
 2   category           114452 non-null  str  
 3   short_description  114452 non-null  str  
 4   authors            114452 non-null  str  
 5   date               114452 non-null  str  
dtypes: str(6)
memory usage: 6.1 MB


In [13]:
data_columns = [column for column in df.columns]
duplicates = scoped_df.loc[scoped_df.duplicated(subset=data_columns, keep=False)].copy()
len(duplicates)

0

In [14]:
audit_df = scoped_df.copy()

for column in ["headline", "short_description", "category"]:
    audit_df[f"_{column}"] = (
        audit_df[column]
        .str.normalize("NFKC")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
audit_df["_headline"] = audit_df["_headline"].str.casefold()
audit_df["_short_description"] = audit_df["_short_description"].str.casefold()
audit_df["_category"] = audit_df["_category"].str.upper()
audit_df.info()

<class 'pandas.DataFrame'>
Index: 114452 entries, 2 to 209512
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype
---  ------              --------------   -----
 0   link                114452 non-null  str  
 1   headline            114452 non-null  str  
 2   category            114452 non-null  str  
 3   short_description   114452 non-null  str  
 4   authors             114452 non-null  str  
 5   date                114452 non-null  str  
 6   _headline           114452 non-null  str  
 7   _short_description  114452 non-null  str  
 8   _category           114452 non-null  str  
dtypes: str(9)
memory usage: 8.7 MB


In [15]:
text_keys = ["_headline", "_short_description"]

duplicate_mask = audit_df.duplicated(
    subset=text_keys + ["_category"],
    keep=False,
)
text_category_duplicates = audit_df.loc[duplicate_mask]
text_category_duplicates

,link,headline,category,short_description,authors,date,_headline,_short_description,_category
17640,https://www.huffingtonpost.com/entry/funniest-...,The Funniest Tweets From Parents This Week,PARENTS,"Kids may say the darndest things, but parents ...",Hollis Miller,2017-12-22,the funniest tweets from parents this week,"kids may say the darndest things, but parents ...",PARENTS
18164,https://www.huffingtonpost.com/entry/funniest-...,The Funniest Tweets From Parents This Week,PARENTS,"Kids may say the darndest things, but parents ...",Hollis Miller,2017-12-15,the funniest tweets from parents this week,"kids may say the darndest things, but parents ...",PARENTS
18974,https://www.huffingtonpost.com/entry/the-shape...,"With This 'Shape Of Water' Clip, You'll Fall F...",ENTERTAINMENT,Spencer and Sally Hawkins play 1960s janitors ...,Matthew Jacobs,2017-12-05,"with this 'shape of water' clip, you'll fall f...",spencer and sally hawkins play 1960s janitors ...,ENTERTAINMENT
19009,https://www.huffingtonpost.com/entry/octavia-s...,"With This 'Shape Of Water' Clip, You'll Fall F...",ENTERTAINMENT,Spencer and Sally Hawkins play 1960s janitors ...,Matthew Jacobs,2017-12-05,"with this 'shape of water' clip, you'll fall f...",spencer and sally hawkins play 1960s janitors ...,ENTERTAINMENT
19064,https://www.huffingtonpost.com/entry/doug-jone...,Abortion Isn’t Main Reason More Republicans St...,POLITICS,Would the Democrats’ Alabama Senate candidate ...,Amanda Terkel,2017-12-04,abortion isn’t main reason more republicans st...,would the democrats’ alabama senate candidate ...,POLITICS
...,...,...,...,...,...,...,...,...,...
208118,https://www.huffingtonpost.com/entry/baftas-20...,BAFTAs 2012 Red Carpet Roundup: Understated Ch...,STYLE & BEAUTY,"It's the eleventh hour for Hollywood style, wi...",Ellie Krupnick,2012-02-12,baftas 2012 red carpet roundup: understated ch...,"it's the eleventh hour for hollywood style, wi...",STYLE & BEAUTY
208219,https://www.huffingtonpost.com/entry/black-lov...,'Black Love' Wedding Announcements (PHOTOS),BLACK VOICES,"""Black Love"" Wedding Announcements: Getting ma...",Julee Wilson,2012-02-11,'black love' wedding announcements (photos),"""black love"" wedding announcements: getting ma...",BLACK VOICES
208636,https://www.huffingtonpost.com/entry/weekly-ro...,Weekly Roundup of eBay Vintage Home Finds (PHO...,HOME & LIVING,"As always, buyer beware! Be sure to read the l...","Mary Kincaid, Contributor\nFounder and Editor ...",2012-02-07,weekly roundup of ebay vintage home finds (pho...,"as always, buyer beware! be sure to read the l...",HOME & LIVING
208789,https://www.huffingtonpost.com/entry/black-lov...,'Black Love' Wedding Announcements (PHOTOS),BLACK VOICES,"""Black Love"" Wedding Announcements: Getting ma...",Julee Wilson,2012-02-05,'black love' wedding announcements (photos),"""black love"" wedding announcements: getting ma...",BLACK VOICES


In [16]:
label_counts = audit_df.groupby(text_keys)["_category"].transform("nunique")
text_label_conflicts = audit_df.loc[label_counts.gt(1)]
text_label_conflicts

,link,headline,category,short_description,authors,date,_headline,_short_description,_category
191206,https://www.huffingtonpost.com/entry/face-oils...,The Great Debate: Face Oils vs. Creams,STYLE & BEAUTY,"If you're looking to have healthier, younger-l...","Hillary Peterson, Contributor\nFounder of True...",2012-08-13,the great debate: face oils vs. creams,"if you're looking to have healthier, younger-l...",STYLE & BEAUTY
191692,https://www.huffingtonpost.com/entry/face-oils...,The Great Debate: Face Oils vs. Creams,WELLNESS,"If you're looking to have healthier, younger-l...","Hillary Peterson, Contributor\nFounder of True...",2012-08-07,the great debate: face oils vs. creams,"if you're looking to have healthier, younger-l...",WELLNESS
192757,https://www.huffingtonpost.com/entry/kelly-wea...,A Conversation With Decorator and Fashion Desi...,HOME & LIVING,Much like the arresting color combinations she...,"Blue Carreon, Contributor\nFashion writer",2012-07-26,a conversation with decorator and fashion desi...,much like the arresting color combinations she...,HOME & LIVING
192783,https://www.huffingtonpost.com/entry/kelly-wea...,A Conversation With Decorator and Fashion Desi...,STYLE & BEAUTY,Much like the arresting color combinations she...,"Blue Carreon, Contributor\nFashion writer",2012-07-26,a conversation with decorator and fashion desi...,much like the arresting color combinations she...,STYLE & BEAUTY
206544,https://www.huffingtonpost.com/entry/yellow-ki...,Stylish Yellow Kitchen Accessories And Appliances,HOME & LIVING,While spring might not be here quite yet -- th...,"Kitchen Daily, Kitchen Daily",2012-02-29,stylish yellow kitchen accessories and appliances,while spring might not be here quite yet -- th...,HOME & LIVING
206666,https://www.huffingtonpost.com/entry/yellow-ki...,Stylish Yellow Kitchen Accessories And Appliances,FOOD & DRINK,While spring might not be here quite yet -- th...,"Kitchen Daily, Kitchen Daily",2012-02-28,stylish yellow kitchen accessories and appliances,while spring might not be here quite yet -- th...,FOOD & DRINK
208182,https://www.huffingtonpost.com/entry/kelly-row...,"Kelly Rowland's Birthday: Singer Turns 31, Sty...",STYLE & BEAUTY,As one of the original members of Destiny's Ch...,Julee Wilson,2012-02-11,"kelly rowland's birthday: singer turns 31, sty...",as one of the original members of destiny's ch...,STYLE & BEAUTY
208217,https://www.huffingtonpost.com/entry/kelly-row...,"Kelly Rowland's Birthday: Singer Turns 31, Sty...",BLACK VOICES,As one of the original members of Destiny's Ch...,Julee Wilson,2012-02-11,"kelly rowland's birthday: singer turns 31, sty...",as one of the original members of destiny's ch...,BLACK VOICES


In [17]:
clean_df = audit_df.copy()
print("before cleaning:", len(clean_df))

rows_before_conflict_removal = len(clean_df)
clean_df = clean_df.loc[~clean_df.index.isin(text_label_conflicts.index)].copy()
conflict_rows_removed = rows_before_conflict_removal - len(clean_df)
print("conflict_rows_removed:", conflict_rows_removed)

rows_before_exact_deduplication = len(clean_df)
clean_df = clean_df.drop_duplicates(subset=data_columns, keep="first")
exact_duplicate_rows_removed = rows_before_exact_deduplication - len(clean_df)
print("exact_duplicate_rows_removed:", exact_duplicate_rows_removed)

rows_before_text_deduplication = len(clean_df)
clean_df = clean_df.drop_duplicates(subset=text_keys + ["_category"], keep="first")
text_duplicate_rows_removed = rows_before_text_deduplication - len(clean_df)
print("text_duplicate_rows_removed:", text_duplicate_rows_removed)

print("after clearning:", len(clean_df))

clean_df = clean_df.reset_index(drop=True)

before cleaning: 114452
conflict_rows_removed: 8
exact_duplicate_rows_removed: 0
text_duplicate_rows_removed: 197
after clearning: 114247


In [18]:
clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 114247 entries, 0 to 114246
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype
---  ------              --------------   -----
 0   link                114247 non-null  str  
 1   headline            114247 non-null  str  
 2   category            114247 non-null  str  
 3   short_description   114247 non-null  str  
 4   authors             114247 non-null  str  
 5   date                114247 non-null  str  
 6   _headline           114247 non-null  str  
 7   _short_description  114247 non-null  str  
 8   _category           114247 non-null  str  
dtypes: str(9)
memory usage: 7.8 MB


In [19]:
clean_df["category"].nunique(), clean_df["_category"].nunique()

(15, 15)

In [20]:
clean_df = clean_df.drop(columns=['_headline', '_short_description', '_category'])

CLEAN_DATA_PATH = Path("data/preprocessed_results.csv")
clean_df.to_csv(CLEAN_DATA_PATH, index=False, encoding="utf-8")

In [21]:
from sklearn.model_selection import train_test_split

TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.20
TEST_RATIO = 0.10
RANDOM_STATE = 42

split_df = clean_df.copy()

In [22]:
train_df, remaining_df = train_test_split(
    split_df,
    train_size=TRAIN_RATIO,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=split_df["category"],
)

In [23]:
validation_df, test_df = train_test_split(
    remaining_df,
    test_size=TEST_RATIO / (VALIDATION_RATIO + TEST_RATIO),
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=remaining_df["category"],
)

In [24]:
split_summary = pd.DataFrame([
    {
        "split": name,
        "rows": len(frame),
        "target_percentage": ratio * 100,
        "actual_percentage": len(frame) / len(split_df) * 100,
    }
    for name, frame, ratio in (
        ("train", train_df, TRAIN_RATIO),
        ("validation", validation_df, VALIDATION_RATIO),
        ("test", test_df, TEST_RATIO),
    )
])
print(split_summary.to_string(index=False))

category_distribution = pd.DataFrame({
    "train": train_df["category"].value_counts(),
    "validation": validation_df["category"].value_counts(),
    "test": test_df["category"].value_counts(),
}).fillna(0).astype(int)
print(category_distribution.to_string())

train_df.to_csv("data/news_train.csv", index=False, encoding="utf-8")
validation_df.to_csv("data/news_val.csv", index=False, encoding="utf-8")
test_df.to_csv("data/news_test.csv", index=False, encoding="utf-8")

     split  rows  target_percentage  actual_percentage
     train 79972               70.0          69.999212
validation 22850               20.0          20.000525
      test 11425               10.0          10.000263
                train  validation  test
category                               
POLITICS        20771        5935  2968
ENTERTAINMENT    9423        2692  1346
WELLNESS         9060        2589  1294
TRAVEL           5615        1604   802
STYLE & BEAUTY   5022        1435   717
PARENTING        4562        1303   652
HEALTHY LIVING   3550        1014   507
QUEER VOICES     3289         940   470
FOOD & DRINK     3168         905   453
BUSINESS         3060         875   437
COMEDY           2754         787   393
SPORTS           2525         722   361
HOME & LIVING    2442         697   349
PARENTS          2415         690   345
BLACK VOICES     2316         662   331
